# T04 Extensions Plugins

Start from a file preset, then **attach extensions in code** (`model_copy` + `extensions`) before `create_sync_logger(config)` — easy to experiment, then freeze in YAML.

- Kernel: `.hydra_env`
- Start Jupyter from the **repo root**, or set `HYDRA_LOGGER_REPO`.
- **§1** sets **repo root** cwd; **§2** imports hydra_logger and runs the scenario.
- Config (when used): `examples/config/dev_console_file.yaml`

## Check

- `model_copy` adds a security extension; password-like fields may be redacted in files.


## 1. Repo setup

Run once: `sys.path` + `chdir` so imports and YAML paths work like the CLI tutorials.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path(os.environ.get("HYDRA_LOGGER_REPO", Path.cwd())).resolve()
_tut = REPO / "examples" / "tutorials"
if not (_tut / "utility").is_dir():
    raise FileNotFoundError(
        "Start Jupyter from the hydra-logger repo root, or set "
        "HYDRA_LOGGER_REPO to that directory (must contain examples/tutorials/utility/)."
    )
sys.path.insert(0, str(_tut))

from utility import notebook_bootstrap

repo_root = notebook_bootstrap()



## 2. Imports + config path

Then load the preset and emit logs (console and/or files under `examples/logs/tutorials/`).

In [ ]:
from hydra_logger import create_sync_logger
from hydra_logger.config.loader import load_logging_config

path = repo_root / "examples" / "config" / "dev_console_file.yaml"
cfg = load_logging_config(path, strict_unknown_fields=True)
cfg = cfg.model_copy(
    update={
        "extensions": {
            "security_redaction": {
                "enabled": True,
                "type": "security",
                "patterns": ["password", "secret"],
            }
        }
    }
)

with create_sync_logger(cfg, strict_unknown_fields=True, name="tutorial-t04") as logger:
    logger.info("login", layer="app", extra={"password": "do-not-log-raw"})


## 3. Customize

Edit `examples/config/`, re-run **§2**. For file output, open paths listed in the intro.